# 09 - Virtual Paper Brokerage & Portfolio Risk Engine

Simulating live execution in paper trading mode: market and limit order lifecycle, position tracking, cash margin, and risk engine constraint enforcement.


In [ ]:
import asyncio

from src.database.session import get_session_factory, init_db
from src.paper.broker import PaperBroker
from src.paper.models import OrderSubmitRequest, PaperOrderSide, PaperOrderType

print('Paper trading broker ready.')


In [ ]:
async def execute_paper_trades():
    await init_db()
    broker = PaperBroker()
    factory = get_session_factory()
    async with factory() as session:
        # Reset virtual account
        acc = await broker.reset_account(session, account_id='research_account', starting_balance=100000.0)
        print(f'Account initialized with ${acc.cash_balance:,.2f}')

        # Submit BUY order for SOL
        order = await broker.submit_order(
            session,
            OrderSubmitRequest(
                account_id='research_account',
                symbol='SOL',
                side=PaperOrderSide.BUY,
                order_type=PaperOrderType.MARKET,
                quantity=50.0,
            )
        )
        print(f'Order {order.id} status: {order.status.value}')
        if order.fills:
            print(f'Filled at ${order.fills[0].fill_price:.2f}')

        port = await broker.get_portfolio(session, account_id='research_account')
        print(f'Current Equity: ${port.total_equity:,.2f}')
        print(f'Open Positions: {[p.symbol for p in port.positions]}')

asyncio.run(execute_paper_trades())
